# 3 - Evaluation and comparison

Validates the retrievals against in-situ drifter velocities (projected onto the line of sight) and compares the **custom pipeline**, the **hybrid** configuration, the operational **Sentinel-1 ocean product**, and the **GLO12** model. Uses the precomputed validation tables in `data/drifters/`.

In [ ]:
import os, sys
from pathlib import Path
# resolve repo root whether launched from notebooks/ or the repo root
REPO = os.getcwd()
if not os.path.exists(os.path.join(REPO, 'scripts', 'sentinel_1')):
    REPO = os.path.abspath(os.path.join(REPO, '..'))
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np
import matplotlib.pyplot as plt
from scripts.run_pipeline import scene_paths, CUSTOM_CFG
from scripts.sentinel_1.grid_merge import merge_burst_grids
print('repo root:', REPO)
import pandas as pd

## Drifter validation: four estimates

In [ ]:
df = pd.read_csv('data/drifters/validation_results.csv')
x = df['v_los_drift'].to_numpy(float)
series = [('v_los_s1_ocn','Custom pipeline','tab:blue'),
          ('v_los_ocn_product','Hybrid (ocean product Doppler)','tab:purple'),
          ('v_los_ocn_native','Sentinel-1 ocean product','tab:orange'),
          ('v_los_glo12','GLO12 model','tab:green')]
def stats(y):
    m = np.isfinite(x) & np.isfinite(y); d = y[m]-x[m]
    r = np.corrcoef(x[m], y[m])[0,1] if m.sum()>1 else np.nan
    return int(m.sum()), d.mean(), np.sqrt((d**2).mean()), r
print(f"{'estimate':30s} {'N':>3} {'bias':>7} {'RMSE':>6} {'r':>6}")
for col, lab, _ in series:
    n,b,rm,r = stats(df[col].to_numpy(float))
    print(f'{lab:30s} {n:3d} {b:+7.3f} {rm:6.3f} {r:+6.3f}')

## Scatter against the drifters

In [ ]:
lim = 1.6
fig, ax = plt.subplots(figsize=(6.4, 6.4), constrained_layout=True)
ax.plot([-lim,lim],[-lim,lim],'--',color='gray',lw=1,label='1:1')
for col, lab, c in series:
    y = df[col].to_numpy(float); n,b,rm,r = stats(y)
    ax.scatter(x, y, s=26, color=c, alpha=0.65, edgecolors='none',
               label=f'{lab}  N={n} bias={b:+.2f} RMSE={rm:.2f} r={r:+.2f}')
ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim); ax.set_aspect('equal')
ax.set_xlabel('drifter radial velocity [m/s]'); ax.set_ylabel('retrieved radial velocity [m/s]')
ax.legend(fontsize=8, loc='upper left'); ax.grid(alpha=0.25); plt.show()

## Cumulative effect of each correction
From the per-burst method sweep (`method_sweep.csv`).

In [ ]:
sw = pd.read_csv('data/drifters/method_sweep.csv')
chain = [('01_geom_last','geometry+sideband'), ('02_geom_stokes_last','+Stokes'),
         ('03_stokes_mouche_last','+wave'), ('04_stokes_mouche_desc_last','+descallop'),
         ('05_full_mouche_last','+mispointing')]
labs, rmses, biases = [], [], []
for key, lab in chain:
    s = sw[sw.method==key]; res = (s.v_los_pipe - s.v_los_drift).to_numpy(float)
    res = res[np.isfinite(res)]; labs.append(lab)
    rmses.append(np.sqrt((res**2).mean())); biases.append(res.mean())
xi = np.arange(len(labs))
fig, ax = plt.subplots(figsize=(8, 4.2), constrained_layout=True)
ax.bar(xi-0.2, rmses, 0.4, color='tab:red', label='RMSE')
ax.bar(xi+0.2, np.abs(biases), 0.4, color='tab:blue', label='|bias|')
ax.set_xticks(xi); ax.set_xticklabels(labs, rotation=15); ax.set_ylabel('error vs drifters [m/s]')
ax.legend(); ax.grid(axis='y', alpha=0.25); ax.set_title('Effect of each correction'); plt.show()

**Reading it:** the custom pipeline beats the operational ocean product on bias, error and correlation; the hybrid is marginally the best SAR-based estimate; GLO12 (assimilating model) is closest to the drifters. Mispointing and the wave correction contribute most; descalloping cleans the image but does not change the drifter statistics.